# 第1章 深度学习导论 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter1/Chapter1.md]]
- [[课程/深度学习/课堂笔记/2026-08-31-第1节-转写_电子科大王君豪的快速会议.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## L1-01 CPU 张量与批量仿射计算

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“PyTorch使用入门及CPU/GPU设置”“Tensor的创建与基本属性”；[[课程/深度学习/课堂笔记/2026-08-31-第1节-转写_电子科大王君豪的快速会议.md]]第六节。新编。


一句话来说：先确定数据的形状、类型、设备，再谈模型。

实现 `tensor_affine(data, weight, bias)`：`data` 为形状 `(N,d)` 的嵌套列表，`weight` 为长 `d` 的列表，`bias` 为数；用 `torch.tensor` 显式建立 **CPU、float64** 张量，返回一维预测张量 `(N,)`，不记录梯度。不得按样本写循环。

用 `data=[[1,2],[-1,0],[0,3]]`、`weight=[2,-1]`、`bias=0.5` 演示，打印预测、shape、dtype、device。输入无随机性，无需种子。教学扩展：读取 Tensor 的 `dtype/device/requires_grad` 属性（课件已出现，不在考试命令表逐项列全）。


<details><summary>提示（先尝试再展开）</summary>

输入矩阵每行是一个样本；`X @ w` 一次得到全批预测，偏置标量广播到各样本。

</details>


In [ ]:
import torch
def tensor_affine(data, weight, bias):
    X = torch.tensor(data, dtype=torch.float64, device='cpu')
    w = torch.tensor(weight, dtype=torch.float64, device='cpu')
    return X @ w + bias
tensor_demo = tensor_affine([[1,2],[-1,0],[0,3]], [2,-1], 0.5)
print(tensor_demo, tensor_demo.shape, tensor_demo.dtype, tensor_demo.device)


### 解析与点评

张量有数值以外的属性。本题不涉及反传或GPU迁移；不能因为安装了CUDA版PyTorch就要求显卡可用。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import torch
_p = tensor_affine([[1,2],[-1,0],[0,3]], [2,-1], 0.5)
assert _p.tolist() == [0.5,-1.5,-2.5]
assert _p.dtype == torch.float64 and _p.device.type == 'cpu'
assert not _p.requires_grad
assert tensor_affine([[3]], [-2], 1).tolist() == [-5.0]

print("自检通过")


## L1-02 把偏置并入设计矩阵

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“机器学习实例：线性回归”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C1-09；[[课程/深度学习/课堂笔记/2026-08-31-第1节-转写_电子科大王君豪的快速会议.md]]“从课件回归例子到可检验的计算”。改编。


实现 `affine_matrix(X)`，输入浮点 NumPy 数组 `(N,d)`，返回 `(N,d+1)`：前 `d` 列为原特征，**最后一列为1**。本章线性参数统一按 `[w_1,...,w_d,b]` 排列。

对 `X=[[0,1],[1,2],[-1,2]]` 与参数 `[2,-0.5,1]` 输出设计矩阵和预测。再对单样本 `(1,2)` 检验接口。不要把多项式常数列在前的约定机械搬来；无随机性。


<details><summary>提示（先尝试再展开）</summary>

`np.column_stack` 把一维常数数组按列接到特征矩阵后。每一列要对应一个参数。

</details>


In [ ]:
import numpy as np
def affine_matrix(X):
    return np.column_stack((X, np.ones_like(X[:,0])))
_X = np.array([[0.,1.],[1.,2.],[-1.,2.]])
print(affine_matrix(_X))
print(affine_matrix(_X) @ np.array([2.,-0.5,1.]))


### 解析与点评

写出 A 的每列含义比只背 shape 更可靠。参数顺序由设计矩阵决定；输入矩阵不应被原地修改。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_X = np.array([[0.,1.],[1.,2.],[-1.,2.]])
_A = affine_matrix(_X)
assert np.array_equal(_A, [[0,1,1],[1,2,1],[-1,2,1]])
assert np.allclose(_A @ np.array([2.,-.5,1.]), [.5,2.,-2.])
assert np.array_equal(affine_matrix(np.array([[1.,2.]])), [[1,2,1]])
assert np.array_equal(_X, [[0,1],[1,2],[-1,2]])

print("自检通过")


## L1-03 逐样本残差与误差口径

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“均方误差”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] A-02、A-03；[[课程/深度学习/课堂笔记/2026-09-07-第3节-机器学习基础与NumPy实操讲座.md]]“自测（新编）”。新编形状防错练习。


实现 `regression_metrics(pred, target)`：输入必须是**相同形状的一维 NumPy 数组**，返回 `(residual, mse, rmse)`，残差定义为 `pred-target`。空数组、二维输入、长度不同均抛出 `ValueError`，不靠广播悄悄运算。

演示 `pred=[2,1,5]`、`target=[1,3,2]`。再说明 `(3,1)` 减 `(3,)` 为什么不是3个残差。误差按样本数除，不用无偏方差的 `N-1`。无随机性。


<details><summary>提示（先尝试再展开）</summary>

一维的维数判断可用 `len(a.shape)`；先验形状，不要在减法以后才检查结果。

</details>


In [ ]:
import numpy as np
def regression_metrics(pred, target):
    if len(pred.shape) != 1 or pred.shape != target.shape or pred.size == 0:
        raise ValueError('expected equal nonempty 1-D arrays')
    residual = pred - target
    mse = float(np.mean(residual**2))
    return residual, mse, float(np.sqrt(mse))
print(regression_metrics(np.array([2.,1.,5.]), np.array([1.,3.,2.])))
print('broadcast shape:', (np.array([[2.],[1.],[5.]])-np.array([1.,3.,2.])).shape)


### 解析与点评

本题训练的是“把公式正确翻译为数组程序”。广播错误往往不报错，必须用具体数值与边界反例检查。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_r,_m,_e = regression_metrics(np.array([2.,1.,5.]),np.array([1.,3.,2.]))
assert np.array_equal(_r,[1,-2,3]) and np.isclose(_m,14/3)
assert np.isclose(_e,np.sqrt(14/3))
assert regression_metrics(np.array([2.]),np.array([2.]))[1:] == (0.,0.)
for _a,_b in [(np.array([]),np.array([])),(np.zeros((3,1)),np.zeros(3)),(np.zeros(2),np.zeros(3))]:
    try:
        regression_metrics(_a,_b)
    except ValueError:
        pass
    else:
        raise AssertionError('非法shape或空数组必须拒绝')

print("自检通过")


## L1-04 最小二乘、噪声与不唯一解

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“机器学习实例：线性回归”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C1-09、A-01。改编。


实现 `fit_affine(X,y)`：`X` 为 `(N,d)`、`y` 为 `(N,)`，返回 `(beta,pred,mse)`；参数顺序为特征权重在前、偏置在后。使用 `np.linalg.lstsq(A,y,rcond=None)[0]`，不显式求逆。

用 `X=[[-2],[-1],[0],[1],[2]]`、`y=[-2,-1,1,3,4]` 拟合并打印参数、预测与MSE。另用完全重复的两个特征检验程序仍可运行：`X=[[0,0],[1,1],[2,2]]`、`y=[1,3,5]`。最小二乘返回最小欧氏范数系数；列相关时不能声称参数唯一。无随机性。


<details><summary>提示（先尝试再展开）</summary>

非零残差不等于算法错。最优处的残差应与设计矩阵各列正交。

</details>


In [ ]:
import numpy as np
def fit_affine(X,y):
    A = np.column_stack((X,np.ones_like(X[:,0])))
    beta = np.linalg.lstsq(A,y,rcond=None)[0]
    pred = A @ beta
    return beta,pred,float(np.mean((pred-y)**2))
print(fit_affine(np.array([[-2.],[-1.],[0.],[1.],[2.]]),np.array([-2.,-1.,1.,3.,4.])))
print('dependent columns:',fit_affine(np.array([[0.,0.],[1.,1.],[2.,2.]]),np.array([1.,3.,5.]))[0])


### 解析与点评

前一数据集并非完全落在直线上，最优MSE为0.08；重复特征的两权重各为1是最小范数选择，不是另一个唯一物理结论。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
_X = np.array([[-2.],[-1.],[0.],[1.],[2.]])
_y = np.array([-2.,-1.,1.,3.,4.])
_b,_p,_m = fit_affine(_X,_y)
assert np.allclose(_b,[1.6,1.]) and np.allclose(_p,[-2.2,-.6,1.,2.6,4.2])
assert np.isclose(_m,.08)
_A = np.column_stack((_X,np.ones(5)))
assert np.linalg.norm(_A.T @ (_p-_y)) < 1e-10
_b,_p,_m = fit_affine(np.array([[0.,0.],[1.,1.],[2.,2.]]),np.array([1.,3.,5.]))
assert np.allclose(_b,[1,1,1]) and _m < 1e-25

print("自检通过")


## L1-05 非线性输入特征仍可线性求系数

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“多项式回归”中 d=2、P=2 的展开；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C1-09 的矩阵思想。新编。


实现 `quadratic_features(X)`，输入 `(N,2)`，输出各列按 `[1,x1,x2,x1²,x1*x2,x2²]` 排列的 `(N,6)` 矩阵。

使用九个点：`x1,x2` 各从 `[-1,0,1]` 取值的全部组合，标签为 `y=1+2*x1-x2+0.5*x1*x2+3*x2²`。求 `quad_coef`，预测新点 `(0.5,-0.5)` 得 `quad_new_pred`。打印并解释为何此问题对输入非线性，却对系数线性。无随机性。


<details><summary>提示（先尝试再展开）</summary>

把每个已知单项式看成新的特征列，未知量仍只是6个系数。常数项本题在首列，不同于前面的仿射参数顺序。

</details>


In [ ]:
import numpy as np
def quadratic_features(X):
    u,v = X[:,0],X[:,1]
    return np.column_stack((np.ones_like(u),u,v,u*u,u*v,v*v))
_X = np.array([[u,v] for u in [-1.,0.,1.] for v in [-1.,0.,1.]])
_y = 1+2*_X[:,0]-_X[:,1]+.5*_X[:,0]*_X[:,1]+3*_X[:,1]**2
quad_coef = np.linalg.lstsq(quadratic_features(_X),_y,rcond=None)[0]
quad_new_pred = float((quadratic_features(np.array([[.5,-.5]])) @ quad_coef)[0])
print(quad_coef,quad_new_pred)


### 解析与点评

是否“线性结构”要看对参数的依赖。不能仅看曲线是否弯曲；交叉项与平方项都是已知输入特征。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
assert np.array_equal(quadratic_features(np.array([[2.,-3.]])),[[1,2,-3,4,-6,9]])
assert np.allclose(quad_coef,[1,2,-1,0,.5,3],atol=1e-12)
assert np.isclose(quad_new_pred,3.125)
_edge = quadratic_features(np.array([[0.,0.],[1.,0.]]))
assert np.array_equal(_edge,[[1,0,0,0,0,0],[1,1,0,1,0,0]])

print("自检通过")


## L1-06 十点带噪指数回归与容量比较

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“示例（d=1）”；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C1-10。保留目标、10点与噪声尺度，补充可复现实验和图表要求的改编。


生成器 `rng=np.random.default_rng(20261001)`，依次生成 `x=rng.uniform(0,1,10)`、`noise=rng.normal(0,0.1,10)`，`y=exp(x)+noise`。**不要改变随机调用顺序**。正态采样 `rng.normal` 为本题教学扩展，非命令表核心要求。

分别用1、3、6次多项式拟合。实现 `exp_capacity(seed=20261001)`，返回 `(x,y,rows,coefs)`：`rows` 为按次数排序的 `(3,2)` 数组，每行 `[degree,train_mse]`；`coefs` 为以次数作键的字典，系数按升幂排列。

调用并打印表；在201点网格画目标、三条拟合曲线及训练点，保存 `output/L1-06.png`。每次先 `Path('output').mkdir(exist_ok=True)`。写两句：训练误差的下降是否足以决定最终模型？本题不以测试数据挑选次数。


<details><summary>提示（先尝试再展开）</summary>

训练损失按噪声标签计算，不应改用真实指数函数值。密集网格只用于画曲线，不是独立测试集。

</details>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
def exp_capacity(seed=20261001):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0,1,10)
    y = np.exp(x)+rng.normal(0,.1,10)
    rows,coefs = [],{}
    for p in [1,3,6]:
        A = np.column_stack([x**j for j in range(p+1)])
        c = np.linalg.lstsq(A,y,rcond=None)[0]
        coefs[p] = c
        rows.append([p,float(np.mean((A@c-y)**2))])
    return x,y,np.array(rows),coefs
exp_x,exp_y,exp_rows,exp_coefs = exp_capacity()
print('degree train_MSE',exp_rows,sep='; ')
Path('output').mkdir(exist_ok=True)
plt.figure()
_g = np.linspace(0,1,201)
plt.plot(_g,np.exp(_g),'k--',label='target')
plt.plot(exp_x,exp_y,'o',label='train')
for _p,_c in exp_coefs.items():
    _G = np.column_stack([_g**j for j in range(_p+1)])
    plt.plot(_g,_G@_c,label=f'degree {_p}')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.savefig('output/L1-06.png'); plt.close()
print('Higher degree lowers the best training loss in nested spaces, but may fit noise.')
print('Independent validation is needed before choosing a final model.')


### 解析与点评

嵌套假设空间的最优训练损失不会变差，但这是充分求解、相同训练数据的结论。六次曲线可能追随噪声，本题不预设哪个模型泛化最好。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_x,_y,_r,_cs = exp_capacity()
_rng = np.random.default_rng(20261001)
_ex = _rng.uniform(0,1,10); _ey = np.exp(_ex)+_rng.normal(0,.1,10)
assert np.allclose(_x,_ex) and np.allclose(_y,_ey)
assert np.array_equal(_r[:,0],[1,3,6]) and np.all(_r[:,1]>=0)
assert np.all(np.diff(_r[:,1]) <= 1e-12)
for _i,_p in enumerate([1,3,6]):
    _A = np.column_stack([_x**j for j in range(_p+1)])
    assert np.isclose(_r[_i,1],np.mean((_A@_cs[_p]-_y)**2))
    assert np.linalg.norm(_A.T@(_A@_cs[_p]-_y)) < 1e-8
assert Path('output/L1-06.png').stat().st_size > 1000

print("自检通过")


## L1-07 完整双特征回归与常数基线

**来源与性质**：[[课程/深度学习/PPT/Chapter1/Chapter1.md]]“机器学习实例：线性回归”；[[课程/深度学习/课堂笔记/2026-08-31-第1节-转写_电子科大王君豪的快速会议.md]]第四、第五节；[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]] C1-09。新编综合实验。


实现 `linear_experiment(seed=20261002)`，在函数内部按以下顺序独立生成数据：`rng.uniform(-1,1,(40,2))` 得训练特征；`rng.uniform(-0.05,0.05,40)` 得训练噪声；`rng.uniform(-1,1,(25,2))` 得测试特征；`rng.uniform(-0.05,0.05,25)` 得测试噪声。目标 `f(X)=2*X[:,0]-0.75*X[:,1]+0.5`，两组标签都加各自噪声。

只用训练集拟合仿射模型与常数基线（常数取训练标签均值）。返回字典，键为 `beta,train_rmse,test_rmse,gap_abs,gap_signed,baseline_test_rmse,test_y,test_pred`。课件泛化差距为 `gap_abs=abs(test_rmse-train_rmse)`，另输出 `gap_signed=test_rmse-train_rmse` 作为诊断；两组误差均对观测标签计算。

打印误差表；画测试真实标签与预测值的对应点和 `y=x` 参考线，保存 `output/L1-07.png`。用两句话解释基线和有限测试集差距的意义。无需PyTorch训练或梯度下降。


<details><summary>提示（先尝试再展开）</summary>

常数基线不能用测试标签均值，否则将测试信息用于拟合。图的横轴应是真实观测标签而非样本索引。

</details>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
def linear_experiment(seed=20261002):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1,1,(40,2)); e = rng.uniform(-.05,.05,40)
    Xt = rng.uniform(-1,1,(25,2)); et = rng.uniform(-.05,.05,25)
    f = lambda a: 2*a[:,0]-.75*a[:,1]+.5
    y,yt = f(X)+e,f(Xt)+et
    A = np.column_stack((X,np.ones(40)))
    At = np.column_stack((Xt,np.ones(25)))
    beta = np.linalg.lstsq(A,y,rcond=None)[0]
    ptr,pt = A@beta,At@beta
    er = float(np.sqrt(np.mean((ptr-y)**2)))
    etest = float(np.sqrt(np.mean((pt-yt)**2)))
    return dict(beta=beta,train_rmse=er,test_rmse=etest,gap_abs=abs(etest-er),gap_signed=etest-er,
                baseline_test_rmse=float(np.sqrt(np.mean((np.mean(y)-yt)**2))),
                test_y=yt,test_pred=pt)
linear_result = linear_experiment()
print({k:v for k,v in linear_result.items() if k not in ['test_y','test_pred']})
Path('output').mkdir(exist_ok=True)
plt.figure()
plt.plot(linear_result['test_y'],linear_result['test_pred'],'o',label='prediction')
_a = np.array([-2.5,3.5]); plt.plot(_a,_a,'k--',label='ideal')
plt.xlabel('observed y'); plt.ylabel('predicted y'); plt.legend()
plt.savefig('output/L1-07.png'); plt.close()
print('The mean-label baseline measures the value of using input features.')
print('The signed gap is a finite-sample estimate, not a guarantee on all unseen data.')


### 解析与点评

基线帮助发现“看似误差小，实际不如不看输入”的程序。测试误差不是必须大于训练误差；课件泛化差距取绝对值，有符号诊断另列。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
from pathlib import Path
_r = linear_experiment()
assert np.allclose(_r['beta'],[2,-.75,.5],atol=.03)
assert 0 < _r['test_rmse'] < .06 and _r['baseline_test_rmse'] > .5
assert np.isclose(_r['test_rmse'],np.sqrt(np.mean((_r['test_pred']-_r['test_y'])**2)))
assert np.isclose(_r['gap_abs'],abs(_r['test_rmse']-_r['train_rmse']))
assert np.isclose(_r['gap_signed'],_r['test_rmse']-_r['train_rmse'])
assert np.allclose(_r['test_pred'],linear_experiment()['test_pred'])
assert not np.allclose(_r['test_y'],linear_experiment(20261003)['test_y'])
assert Path('output/L1-07.png').stat().st_size > 1000

print("自检通过")
